# 구글 뉴스 검색 방식 시험

> **시험용 노트북이다.** 검색 방식(매체, OR 묶음, 기간 조건, 거르기 규칙, 검색어 모양)을 정하려고 한 시험 기록이다. 실제 수집은 `../news_search/news_collect.ipynb`, 처리는 `../news_search/news_process.ipynb`가 한다. 새 검색 방식 시험은 여기에 절을 더하고, 정한 결과만 두 노트북과 `news_search.md`에 옮긴다. (2026-11-16에 `news_search.ipynb`에서 이름을 바꾸고, 같은 날 시험 자료와 함께 `news_search/`에서 이 폴더로 옮김)

`graph_db/search_terms.ipynb`가 만든 검색어로 **구글 뉴스 RSS(영어, 미국판)**를 검색해 밤사이 미국 기사를 모은다. 결정과 근거는 `../news_search/news_search.md`, 검색어 규칙은 `graph_db/graphDB.md` 6.5. 검색어를 만드는 일은 `graph_db`가, 검색해서 기사를 모으는 일은 `news_search/`가 맡는다. 이 폴더는 검색 방식 시험만 둔다.

- **본문을 확보할 수 있는 매체만** `site:`로 제한해 검색한다. 제한하지 않으면 결과 100건 한도 안에 본문을 못 받는 매체(Reuters, Bloomberg 등)가 섞여 쓸 수 있는 기사가 밀려난다.
- **OR 묶음**: 매체 11곳을 `(site:a OR site:b OR …)`로 묶어 검색어 하나에 요청 1건으로 검색한다. 결과가 100건이면(한도) 그 검색어만 매체별로 나눠 다시 검색한다. 매체별로만 검색하면 요청이 11배다.
- 응답은 원본 그대로 저장하고(검색 문자열의 해시 이름), 저장된 것은 다시 요청하지 않는다. 요청 간격은 6초(2.5초로 약 700건을 보내자 503으로 막혔다). 429·503이면 20분 쉬었다가 다시 하고(최대 6번), 계속 막히면 멈추고 텔레그램으로 알린다.
- 지금은 **시험 단계**다: 1차 시험(매체 제한 없음) 결과 보기, OR 묶음 확인, 2차 시험(본문 확보 매체), 기사 아닌 페이지 거르기, 3차 시험(2단어 중심 검색어). 서비스용 검색(보유 종목의 검색어, 밤사이 하루)은 시험 결과로 규칙을 정한 뒤 만든다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `../graph_db/source/search/test_queries.csv` | 시험 검색어 목록(종목, 검색어 종류, 모양, 검색 문자열). 시험을 처음 할 때 그 시험 폴더에 복사해 고정하고, 이후에는 고정한 사본을 읽는다(검색어 규칙이 바뀌어도 끝난 시험은 그대로) |
| `../graph_db/source/search/company_types.csv` | 종목 유형, 연결이 적은 종목 |
| `../news_source_tests/google_rss_sources/round1·2/source_summary.csv` | 언론사 시험: 매체마다 본문을 가져올 수 있는지(USABLE 등) |

**출력** (`source/`)

| 폴더·파일 | 내용 |
|---|---|
| `rss_test_unrestricted/` | 1차 시험(매체 제한 없음, 11-12~14): 응답 원본(`raw/`), 검색 문자열별 기사 수, 기사 목록 |
| `or_check/` | OR 묶음 확인: 응답 원본, 비교 결과 |
| `rss_test_usable/` | 2차 시험(본문 확보 매체 11곳, 야후 1건 + 나머지 OR 1건): 응답 원본, 검색어별 기사 수, 기사 목록, 거른 뒤 기사 |
| `rss_test_short/` | 3차 시험(2단어 중심 검색어, 같은 매체·기간·방식): 2차에 없던 검색 문자열의 응답 원본, 검색어별 기사 수, 기사 목록, 거른 뒤 기사 |
| `<시험 폴더>/test_queries.csv` | 그 시험에 쓴 검색어 목록 사본 |
| `live_check/`, `cluster_test/`, `morning_시험_20261116/` | 노트북 밖에서 한 확인의 결과: `when:` 실시간 확인, 제목 임베딩 묶기 시험(`news_process.ipynb` 13절이 읽음), 첫 밤 처리 흉내 |

3차 시험 목록(`rss_test_short/test_queries.csv`, `queries.csv`)은 수집·처리 노트북이 임시 검색어 표로 쓰고 있어 `../news_search/source/terms/`에 사본을 두었다. 전체 검색어를 확정하면 실제 검색어 표로 바꾼다.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `graph_db/search_terms`(검색어) → **`news_search`**(검색) → 본문 받기 → 매일 밤 프롬프트 |
| 외부 요청·비용 | 구글 뉴스 RSS 웹 요청(비용 없음). `RUN = True`일 때만 보내고, 건수·시간을 알리고 승인받은 뒤 한다 |
| 다시 실행 | 언제든 된다. 받은 응답은 다시 요청하지 않는다 |

## 1. 설정

In [1]:
# ── 설정 ──
import gzip
import hashlib
import random
import shutil
import time
from collections import Counter
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'news_search_tests', 'news_search_tests 폴더에서 실행해야 한다.'
PROJ = ROOT.parent
SEARCH = PROJ / 'graph_db' / 'source' / 'search'      # 검색어(graph_db/search_terms)
OUT = ROOT / 'source'
OUT.mkdir(exist_ok=True)
RUN = False                 # True로 바꿔야 요청을 보낸다
RSS_SLEEP = 6               # 요청 간격(초, ± 0.5 무작위). 2.5초로 약 700건(1시간 10분)을 보내자 503으로 막혔다(2026-11-15)
BLOCK_WAIT = 1200           # 429·503으로 막히면 이만큼(초) 쉬고 다시 시도
MAX_BLOCK_WAITS = 6         # 막혔을 때 쉬는 최대 횟수(넘으면 멈춘다. 다시 실행하면 남은 것부터)
CAP = 100                   # 구글 뉴스 RSS가 한 번에 돌려주는 최대 기사 수
ET = ZoneInfo('America/New_York')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36',
           'Accept': 'application/rss+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-US,en;q=0.9'}
text_hash = lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16]


def save_csv(df, path):
    """내용이 바뀔 때만 쓴다(iCloud 사본 방지)."""
    text = df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8-sig') == text:
        return '그대로'
    path.write_text(text, encoding='utf-8-sig')
    return '씀'

## 2. 검색할 매체

언론사 시험(`news_source_tests/1_google_rss_source_test.ipynb`, round1·2)에서 본문을 가져올 수 있다고 판정한 곳(USABLE 8곳)과, 표본은 모자라지만 본문 추출이 모두 성공한 곳(3곳)이다. Reuters·Bloomberg·WSJ·AP·Barron's·MarketWatch·Seeking Alpha·Forbes·Benzinga 등은 차단·유료라 뺀다.

In [2]:
# ── 검색할 매체: 언론사 시험 판정에서 ──
v = pd.concat([pd.read_csv(PROJ / 'news_source_tests' / 'google_rss_sources' / r / 'source_summary.csv') for r in ('round1', 'round2')]).drop_duplicates('source', keep='last')
usable = v[(v['verdict'] == 'USABLE') | ((v['verdict'] == 'INSUFFICIENT_SAMPLE') & (v['article_success_rate'] == 1))]
SITES = dict(zip(usable['source'], usable['source_name']))
VERDICT = dict(zip(v['source'], v['verdict']))
print(f'검색할 매체 {len(SITES)}곳:', ', '.join(f'{n}({d})' for d, n in SITES.items()))

검색할 매체 11곳: Yahoo Finance(finance.yahoo.com), CNBC(cnbc.com), CBS News(cbsnews.com), New York Post(nypost.com), The Motley Fool(fool.com), Fortune(fortune.com), The Guardian(theguardian.com), USA Today(usatoday.com), Business Insider(businessinsider.com), CNN Business(cnn.com), Fox Business(foxbusiness.com)


## 3. 검색 도구

- 검색 문자열: `검색어 [매체 제한] after:시작일 before:끝일`. 매체 제한은 없음 / 매체 하나(`site:a`) / OR 묶음(`(site:a OR site:b …)`).
- 응답 저장: `source/<폴더>/raw/<해시>.xml.gz`. 저장된 것은 다시 요청하지 않는다.
- 기사: 제목, 매체(출처 주소의 도메인), 발행 시각(미국 동부), 구글 뉴스 링크.

In [3]:
# ── 검색 도구: 검색 문자열 만들기, 요청·저장(있으면 건너뜀), 응답 읽기 ──
def site_clause(sites):
    if not sites:
        return ''
    return f'site:{sites[0]}' if len(sites) == 1 else '(' + ' OR '.join(f'site:{s}' for s in sites) + ')'


def full_query(term, sites, after, before):
    return ' '.join(x for x in [term, site_clause(sites), f'after:{after}', f'before:{before}'] if x)


rss_url = lambda q: 'https://news.google.com/rss/search?q=' + quote_plus(q) + '&hl=en-US&gl=US&ceid=US:en'


def raw_path(folder, q):
    return OUT / folder / 'raw' / f'{text_hash(q)}.xml.gz'


def notify(text):
    """텔레그램으로 알린다(프로젝트 루트 .env의 ALERT_TELEGRAM_TOKEN·CHAT_ID). 실패해도 멈추지 않고, 오류 메시지에서 토큰을 가린다."""
    env = {}
    for line in (PROJ / '.env').read_text(encoding='utf-8-sig').splitlines():
        k, _, v = line.partition('=')
        env[k.strip()] = v.strip().strip('"\'')
    token, chat = env.get('ALERT_TELEGRAM_TOKEN'), env.get('ALERT_TELEGRAM_CHAT_ID')
    print('알림:', text)
    if token and chat:
        try:
            requests.post(f'https://api.telegram.org/bot{token}/sendMessage', data={'chat_id': chat, 'text': text}, timeout=20)
        except requests.RequestException as e:
            print('  텔레그램 알림 실패:', str(e).replace(token, '***')[:200])


def fetch_all(folder, queries):
    """검색 문자열들을 요청해 저장한다. 이미 받은 것은 건너뛴다. 429·503이면 BLOCK_WAIT초 쉬고 다시(최대 MAX_BLOCK_WAITS번), 그래도 막히면 멈춘다."""
    (OUT / folder / 'raw').mkdir(parents=True, exist_ok=True)
    todo = [q for q in dict.fromkeys(queries) if not raw_path(folder, q).exists()]
    print(f'{folder}: 검색 {len(set(queries))}개 / 받은 것 {len(set(queries)) - len(todo)} / 남은 것 {len(todo)} (예상 {len(todo) * (RSS_SLEEP + 0.4) / 60:.0f}분)')
    if not RUN or not todo:
        if not RUN and todo:
            print('RUN = False: 요청하지 않았다.')
        return
    ses = requests.Session()
    ses.headers.update(HEADERS)
    t0, n_ok, waits = time.time(), 0, 0
    for i, q in enumerate(todo):
        code_ = None
        while True:
            try:
                resp = ses.get(rss_url(q), timeout=30)
                code_ = resp.status_code
            except requests.RequestException as e:
                resp, code_ = None, type(e).__name__
            if code_ == 200:
                raw_path(folder, q).write_bytes(gzip.compress(resp.content))
                n_ok += 1
                break
            if code_ in (429, 503) and waits < MAX_BLOCK_WAITS:
                waits += 1
                print(f'  {time.strftime("%H:%M")} {code_}로 막힘({i}/{len(todo)}) → {BLOCK_WAIT // 60}분 쉬고 다시({waits}/{MAX_BLOCK_WAITS})')
                time.sleep(BLOCK_WAIT)
                continue
            print(f'  {q[:70]}: {code_}')
            break
        if code_ in (429, 503):
            notify(f'[news_search] {folder}: 구글 뉴스 RSS가 계속 막혀 멈췄다({n_ok}/{len(todo)}건 받음). 나중에 다시 실행하면 남은 것부터 한다.')
            break
        if (i + 1) % 50 == 0:
            print(f'  {i + 1}/{len(todo)} ({time.time() - t0:.0f}초)')
        time.sleep(RSS_SLEEP + random.uniform(-0.5, 0.5))
    print(f'  받음 {n_ok}/{len(todo)}, {time.time() - t0:.0f}초, 막혀서 쉰 횟수 {waits}')
    if n_ok == len(todo):
        notify(f'[news_search] {folder}: 요청 {n_ok}건 모두 받음({(time.time() - t0) / 60:.0f}분, 막혀서 쉰 횟수 {waits}).')


def read_items(folder, q):
    """저장한 응답 하나 → 기사 목록(없으면 None)."""
    p = raw_path(folder, q)
    if not p.exists():
        return None
    out = []
    for e in feedparser.parse(gzip.decompress(p.read_bytes())).entries:
        try:
            d = parsedate_to_datetime(e.get('published')).astimezone(ET)
        except Exception:
            d = None
        src = e.get('source') or {}
        dom = src.get('href', '').replace('https://', '').replace('http://', '').replace('www.', '').strip('/')
        out.append(dict(title=e.get('title', ''), source=src.get('title', ''), domain=dom, published=d.isoformat() if d else '',
                        day=d.strftime('%m-%d') if d else '', link=e.get('link', '')))
    return out


def frozen_testq(folder):
    """시험에 쓴 검색어 목록을 그 시험 폴더에 고정한다: 처음 한 번 graph_db의 최신 목록을 복사하고, 이후에는 그 사본을 읽는다."""
    p = OUT / folder / 'test_queries.csv'
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(SEARCH / 'test_queries.csv', p)
        print(f'{folder}: graph_db의 최신 시험 목록을 복사해 고정했다')
    return pd.read_csv(p, dtype=str).fillna('')

## 4. 1차 시험 다시 보기: 매체 제한 없음

`graph_db`에서 한 1차 시험(검색 문자열 309개, 11-12~14, 매체 제한 없음)의 저장된 응답을 다시 읽는다(요청 없음). 검색 문자열은 그때 형식(`검색어 after:… before:…`)이다.

- 결과 기사의 매체를 언론사 시험 판정과 대조한다: 본문을 가져올 수 있는 매체에서 온 기사가 얼마인지.

In [4]:
# ── 1차 시험(매체 제한 없음): 저장된 응답으로 매체 판정 대조 ──
FIRST = 'rss_test_unrestricted'
TESTQ = frozen_testq(FIRST)
first_items = []
for q in dict.fromkeys(TESTQ['query']):
    its = read_items(FIRST, full_query(q, [], '2026-11-11', '2026-11-15'))
    if its is not None:
        first_items += [dict(it, query=q) for it in its]
FI = pd.DataFrame(first_items)
FI['verdict'] = FI['domain'].map(VERDICT).fillna('시험 안 함')
FI.loc[FI['domain'].isin(SITES), 'verdict'] = '검색할 매체'
u = FI.drop_duplicates('title')
print(f"기사가 1건 이상인 응답 {FI['query'].nunique()}개 / 기사 {len(u):,}건(제목 중복 제거), 도메인 {u['domain'].nunique():,}곳")
print('매체 판정별 기사', {k: f'{n} ({n / len(u):.0%})' for k, n in u['verdict'].value_counts().items()})
dom = u['domain'].value_counts()
print('상위 도메인이 차지하는 비율', {k: f'{dom.head(k).sum() / len(u):.0%}' for k in (20, 50, 100, 200)})

기사가 1건 이상인 응답 174개 / 기사 3,881건(제목 중복 제거), 도메인 1,563곳
매체 판정별 기사 {'시험 안 함': '3548 (91%)', '검색할 매체': '190 (5%)', 'NOT_USABLE': '143 (4%)'}
상위 도메인이 차지하는 비율 {20: '22%', 50: '34%', 100: '45%', 200: '55%'}


## 5. OR 묶음 확인

검색어 10개를 (a) 매체 11곳 OR 묶음 1건, (b) 매체별 11건으로 검색해 결과를 비교한다. 같은 기간(11-12~14, 1차 시험과 같음).

- OR 묶음 결과가 매체별 결과의 합과 같으면(기사 링크 기준), 서비스에서 OR 묶음을 먼저 쓰고 한도에 걸린 검색어만 매체별로 나눈다.
- 검색어는 기사 수가 많을 것과 적을 것을 섞었다(회사 이름, 대형 해외 상대, 2단어 산업 검색어).
- 요청: 10 × (1 + 11) = 120건, 약 5분.

In [5]:
# ── OR 묶음 확인: OR 묶음 1건 vs 매체별 11건 ──
CHECK = 'or_check'
CHECK_AFTER, CHECK_BEFORE = '2026-11-11', '2026-11-15'
CHECK_TERMS = ['"SK hynix"', '"Nvidia"', '"Eli Lilly"', '"HBM demand"', '"memory chip prices"', '"LNG carrier orders"', '"K-beauty"',
               '"GLP-1 drugs"', '"crude oil prices"', '"defense spending"']
sites = list(SITES)
plan = []
for t in CHECK_TERMS:
    plan.append(dict(term=t, mode='OR 묶음', site='', q=full_query(t, sites, CHECK_AFTER, CHECK_BEFORE)))
    plan += [dict(term=t, mode='매체별', site=s, q=full_query(t, [s], CHECK_AFTER, CHECK_BEFORE)) for s in sites]
PLAN = pd.DataFrame(plan)
print('예:', PLAN['q'].iloc[0])
fetch_all(CHECK, PLAN['q'])

예: "SK hynix" (site:finance.yahoo.com OR site:cnbc.com OR site:cbsnews.com OR site:nypost.com OR site:fool.com OR site:fortune.com OR site:theguardian.com OR site:usatoday.com OR site:businessinsider.com OR site:cnn.com OR site:foxbusiness.com) after:2026-11-11 before:2026-11-15
or_check: 검색 120개 / 받은 것 120 / 남은 것 0 (예상 0분)


In [6]:
# ── OR 묶음 확인 결과: 검색어마다 OR 묶음 기사 수, 매체별 합, 겹침, 다른 매체가 섞였는지 ──
rows = []
for t, g in PLAN.groupby('term', sort=False):
    or_items = read_items(CHECK, g.loc[g['mode'] == 'OR 묶음', 'q'].iloc[0])
    per = [read_items(CHECK, q) for q in g.loc[g['mode'] == '매체별', 'q']]
    if or_items is None or any(x is None for x in per):
        continue
    or_links = {it['link'] for it in or_items}
    per_links = {it['link'] for its in per for it in its}
    per_titles = {it['title'] for its in per for it in its}
    rows.append(dict(term=t, OR_묶음=len(or_items), 매체별_합=len(per_links), 매체별_최대=max(len(x) for x in per),
                     겹침_링크=len(or_links & per_links), 겹침_제목=len({it['title'] for it in or_items} & per_titles),
                     OR에만=len(or_links - per_links), 매체별에만=len(per_links - or_links),
                     다른_매체=sum(it['domain'] not in SITES for it in or_items), 한도=len(or_items) >= CAP))
CMP = pd.DataFrame(rows)
if len(CMP):
    display(CMP)
    print(f"OR 묶음 합 {CMP['OR_묶음'].sum()} / 매체별 합 {CMP['매체별_합'].sum()} / 제목으로 겹친 비율 {CMP['겹침_제목'].sum() / max(CMP['매체별_합'].sum(), 1):.0%} / "
          f"OR 결과에 다른 매체 {CMP['다른_매체'].sum()}건")
    print('compare.csv', save_csv(CMP, OUT / CHECK / 'compare.csv'))
else:
    print('아직 결과가 없다.')

,term,OR_묶음,매체별_합,매체별_최대,겹침_링크,겹침_제목,OR에만,매체별에만,다른_매체,한도
0,"""SK hynix""",100,149,100,38,39,62,111,6,True
1,"""Nvidia""",100,926,100,100,100,0,826,0,True
2,"""Eli Lilly""",100,148,100,28,28,72,120,6,True
3,"""HBM demand""",55,4,4,2,2,53,2,14,False
4,"""memory chip prices""",65,8,7,5,5,60,3,6,False
5,"""LNG carrier orders""",3,0,0,0,0,3,0,0,False
6,"""K-beauty""",73,29,20,9,9,64,20,0,False
7,"""GLP-1 drugs""",100,60,35,11,11,89,49,2,True
8,"""crude oil prices""",100,88,76,17,18,83,71,9,True
9,"""defense spending""",100,69,51,12,13,88,57,5,True


OR 묶음 합 796 / 매체별 합 1481 / 제목으로 겹친 비율 15% / OR 결과에 다른 매체 48건
compare.csv 그대로


### 5-2. OR 묶음 문법 비교와 야후 따로

5절 결과, `검색어 (site:… OR …) after:… before:…` 모양은 **기간 조건이 무시되어** 100건 자리를 옛 기사(2014~2026년 상반기)가 차지했다. 기간 안 기사만 보면 매체별 검색 결과의 일부였다(GLP-1 drugs: 13 vs 53).

- 문법 3가지 비교(검색어 3개): A 날짜를 먼저(`검색어 after:… before:… (site:… OR …)`), B 괄호 없음, C `when:3d`. **A만 기간이 지켜졌다**(GLP-1 drugs 기간 안 52, defense spending 56).
- A로도 빠지는 기사는 대부분 Yahoo Finance였다(야후는 기사가 가장 많고 지역판 주소가 섞인다). 그래서 **야후는 따로 1건, 나머지 10곳은 A 모양 OR 묶음 1건**, 검색어당 요청 2건으로 다시 비교했다(검색어 10개).

In [7]:
# ── 5-2: 문법 비교(검색어 3개 × A·B·C), 야후 따로 + 나머지 OR(검색어 10개) ──
in_window = lambda it: CHECK_AFTER <= it['published'][:10] <= CHECK_BEFORE
orr = lambda ss: '(' + ' OR '.join(f'site:{x}' for x in ss) + ')'
VARIANTS = {'A 날짜 먼저': lambda t: f'{t} after:{CHECK_AFTER} before:{CHECK_BEFORE} {orr(sites)}',
            'B 괄호 없음': lambda t: f"{t} {' OR '.join(f'site:{x}' for x in sites)} after:{CHECK_AFTER} before:{CHECK_BEFORE}",
            'C when:3d': lambda t: f'{t} {orr(sites)} when:3d'}
rest = [x for x in sites if x != 'finance.yahoo.com']
yahoo_q = lambda t: full_query(t, ['finance.yahoo.com'], CHECK_AFTER, CHECK_BEFORE)
rest_q = lambda t: f'{t} after:{CHECK_AFTER} before:{CHECK_BEFORE} {orr(rest)}'
VAR_TERMS = ['"GLP-1 drugs"', '"defense spending"', '"SK hynix"']
fetch_all(CHECK, [f(t) for t in VAR_TERMS for f in VARIANTS.values()] + [q for t in CHECK_TERMS for q in (yahoo_q(t), rest_q(t))])
rows = []
for t in VAR_TERMS:
    for name, f in VARIANTS.items():
        its = read_items(CHECK, f(t)) or []
        rows.append(dict(term=t, 문법=name, 전체=len(its), 기간_안=sum(map(in_window, its)), 가장_옛날=min((it['published'][:10] for it in its), default='')))
display(pd.DataFrame(rows))
rows = []
for t in CHECK_TERMS:
    per = {it['title'] for x in sites for it in (read_items(CHECK, full_query(t, [x], CHECK_AFTER, CHECK_BEFORE)) or []) if in_window(it)}
    y, o = read_items(CHECK, yahoo_q(t)) or [], read_items(CHECK, rest_q(t)) or []
    yo = {it['title'] for it in y + o if in_window(it)}
    rows.append(dict(term=t, 매체별_11건=len(per), 야후=len(y), 나머지_OR=len(o), 합_기간_안=len(yo), 겹침=len(yo & per),
                     재현율=round(len(yo & per) / max(len(per), 1), 2), 한도=max(len(y), len(o)) >= CAP))
YO = pd.DataFrame(rows)
display(YO)
ok = YO[~YO['한도'] & (YO['매체별_11건'] > 0)]
print(f"한도에 안 걸린 검색어의 재현율 {ok['겹침'].sum() / ok['매체별_11건'].sum():.0%} (검색어 {len(ok)}개) / 한도에 걸린 검색어 {YO['한도'].sum()}개는 매체별·날짜별로 나눠야 한다")
print('yahoo_rest.csv', save_csv(YO, OUT / CHECK / 'yahoo_rest.csv'))

or_check: 검색 29개 / 받은 것 29 / 남은 것 0 (예상 0분)


,term,문법,전체,기간_안,가장_옛날
0,"""GLP-1 drugs""",A 날짜 먼저,58,52,2022-11-10
1,"""GLP-1 drugs""",B 괄호 없음,100,11,2018-08-03
2,"""GLP-1 drugs""",C when:3d,100,11,2018-08-03
3,"""defense spending""",A 날짜 먼저,57,56,2026-11-05
4,"""defense spending""",B 괄호 없음,100,13,2014-03-25
5,"""defense spending""",C when:3d,100,13,2014-03-25
6,"""SK hynix""",A 날짜 먼저,100,75,2017-05-23
7,"""SK hynix""",B 괄호 없음,100,39,2016-07-07
8,"""SK hynix""",C when:3d,100,39,2016-07-07


,term,매체별_11건,야후,나머지_OR,합_기간_안,겹침,재현율,한도
0,"""SK hynix""",118,100,30,100,100,0.85,True
1,"""Nvidia""",916,100,100,199,198,0.22,True
2,"""Eli Lilly""",108,100,27,97,97,0.90,True
3,"""HBM demand""",4,4,0,4,4,1.00,False
4,"""memory chip prices""",8,7,1,8,8,1.00,False
5,"""LNG carrier orders""",0,0,0,0,0,0.00,False
6,"""K-beauty""",21,20,7,19,19,0.90,False
7,"""GLP-1 drugs""",53,35,23,51,51,0.96,False
8,"""crude oil prices""",88,76,8,84,84,0.95,False
9,"""defense spending""",68,51,16,66,66,0.97,False


한도에 안 걸린 검색어의 재현율 96% (검색어 6개) / 한도에 걸린 검색어 3개는 매체별·날짜별로 나눠야 한다
yahoo_rest.csv 그대로


## 6. 2차 시험: 본문 확보 매체만 (야후 1건 + 나머지 OR 1건)

1차 시험과 같은 검색어 309개·같은 기간(11-12~14)을 2절의 매체 11곳으로만 검색한다. 검색어마다 요청 2건(5-2절 방식).

- 결과가 100건이면(한도) 나눠서 다시 검색한다: 야후는 하루씩(3건), 나머지 OR은 매체별(10건).
- 기간은 발행 시각(미국 동부)으로 한 번 더 거른다(OR 묶음에 옛 기사가 몇 건 섞인다).
- 기사는 제목으로 중복을 없앤다(야후 지역판처럼 같은 기사가 여러 주소로 온다).
- 요청: 약 620건 + 나누는 몫. 첫 실행은 257건을 받은 뒤 503으로 막혀(오늘 누적 약 700건, 간격 2.5초), 간격 6초와 막히면 20분 쉬기로 바꿔 이어서 받았다. `RUN = True`일 때만.

In [8]:
# ── 2차 시험: 검색어마다 야후 1건 + 나머지 10곳 OR 1건, 한도면 나눠서 ──
SECOND = 'rss_test_usable'
TESTQ = frozen_testq(SECOND)                        # 2차 시험 목록(1차와 같다)
T_AFTER, T_BEFORE = '2026-11-11', '2026-11-15'
DAYS = ['2026-11-11', '2026-11-12', '2026-11-13', '2026-11-14', '2026-11-15']
y_q = lambda t, a=T_AFTER, b=T_BEFORE: full_query(t, ['finance.yahoo.com'], a, b)
r_q = lambda t: f'{t} after:{T_AFTER} before:{T_BEFORE} {orr(rest)}'
terms2 = list(dict.fromkeys(TESTQ['query']))
fetch_all(SECOND, [q for t in terms2 for q in (y_q(t), r_q(t))])
# 한도에 걸린 것만 나누기: 야후는 하루씩, 나머지 OR은 매체별
split = []
for t in terms2:
    y, o = read_items(SECOND, y_q(t)), read_items(SECOND, r_q(t))
    if y is not None and len(y) >= CAP:
        split += [y_q(t, a, b) for a, b in zip(DAYS[:-1], DAYS[1:])]
    if o is not None and len(o) >= CAP:
        split += [full_query(t, [x], T_AFTER, T_BEFORE) for x in rest]
print(f'한도에 걸려 나눌 요청 {len(split)}건')
fetch_all(SECOND, split)

rss_test_usable: 검색 618개 / 받은 것 618 / 남은 것 0 (예상 0분)


한도에 걸려 나눌 요청 118건
rss_test_usable: 검색 118개 / 받은 것 118 / 남은 것 0 (예상 0분)


In [9]:
# ── 2차 시험 결과 모으기: 검색어마다 기간 안 기사(제목 중복 제거) ──
in_window2 = lambda it: T_AFTER <= it['published'][:10] <= T_BEFORE


def split_queries(folder, t):
    """한도(100건)에 걸린 검색어만 나누기: 야후는 하루씩, 나머지 OR은 매체별."""
    y, o = read_items(folder, y_q(t)), read_items(folder, r_q(t))
    out = []
    if y is not None and len(y) >= CAP:
        out += [y_q(t, a, b) for a, b in zip(DAYS[:-1], DAYS[1:])]
    if o is not None and len(o) >= CAP:
        out += [full_query(t, [x], T_AFTER, T_BEFORE) for x in rest]
    return out


def gather(folder, t):
    """검색어 하나의 기간 안·검색 매체 기사(제목 중복 제거) → (기사 dict, 한도에 걸렸는지). 아직 안 받았으면 None."""
    y, o = read_items(folder, y_q(t)), read_items(folder, r_q(t))
    if y is None or o is None:
        return None
    parts = [y, o] + [read_items(folder, q) or [] for q in split_queries(folder, t)]
    seen = {}
    for it in (it for p in parts for it in p if in_window2(it) and (it['domain'] in SITES or any(it['domain'].endswith('.' + d) for d in SITES))):
        seen.setdefault(it['title'], it)
    return seen, len(y) >= CAP or len(o) >= CAP


rows2, items2 = [], []
for t in terms2:
    g = gather(SECOND, t)
    if g is None:
        continue
    seen, capped = g
    items2 += [dict(it, query=t) for it in seen.values()]
    rows2.append(dict(query=t, n=len(seen), capped=capped, yahoo=sum('yahoo' in it['domain'] for it in seen.values())))
RQ2, RI2 = pd.DataFrame(rows2), pd.DataFrame(items2)
if len(RQ2):
    print(f"검색어 {len(RQ2)}/{len(terms2)} / 기사 {RI2['title'].nunique():,}건(제목 중복 제거) / 0건 {(RQ2['n'] == 0).sum()}개 / 한도에 걸려 나눈 것 {RQ2['capped'].sum()}개 / "
          f"야후 비중 {RQ2['yahoo'].sum() / max(RQ2['n'].sum(), 1):.0%}")
    print('queries.csv', save_csv(RQ2, OUT / SECOND / 'queries.csv'), '/ items.csv', save_csv(RI2, OUT / SECOND / 'items.csv'))
else:
    print('아직 결과가 없다.')

검색어 309/309 / 기사 6,833건(제목 중복 제거) / 0건 171개 / 한도에 걸려 나눈 것 15개 / 야후 비중 57%
queries.csv 그대로 / items.csv 그대로


## 7. 2차 시험 결과 보기

본문을 확보할 수 있는 매체 11곳의 기간 안 기사(제목 중복 제거)로 센다. 1차 시험(매체 제한 없음)과 같은 검색어·기간이다.

- **종류·모양별**: 검색 문자열 수, 3일 기사 수 중앙값, 0건 비율.
- **단어 수별**(주제만 모양): 검색어가 몇 단어일 때 기사가 잡히는지.
- **종목별**: 회사 고유 검색어(자기 회사·제품·브랜드·해외 상대·원자재·정책)와 묶음 검색어(업종·테마·회사 기본, 테마 대표)로 잡힌 기사 수.
- **제목 표본**: 종류마다 검색어 몇 개의 상위 제목.

In [10]:
# ── 2차 시험 결과: 종류·모양별, 단어 수별, 종목별, 제목 표본 ──
R2 = TESTQ.merge(RQ2[['query', 'n', 'capped']], on='query', how='left')
R2['n'] = R2['n'].fillna(-1).astype(int)
uq = R2.drop_duplicates(['kind', 'shape', 'query'])
print('■ 종류·모양별(기간 안, 본문 확보 매체)')
display(uq.groupby(['kind', 'shape']).agg(검색=('query', 'size'), 기사수_중앙값=('n', 'median'), 영건_비율=('n', lambda s: round((s == 0).mean(), 2))))
t = R2[R2['shape'] == '주제만'].drop_duplicates('query').copy()
t['단어'] = t['term'].str.split().str.len().clip(upper=5)
print('■ 단어 수별(주제만)')
display(t.groupby('단어').agg(검색=('query', 'size'), 기사수_중앙값=('n', 'median'), 영건_비율=('n', lambda s: round((s == 0).mean(), 2))))
own = {'자기 회사', '구체 제품', '넓은 제품', '브랜드', '해외 상대', '원자재·지표', '정책'}
RIx2 = RI2.merge(TESTQ[['query', 'company', 'kind']], on='query')
RIx2['from'] = RIx2['kind'].map(lambda k: '회사 고유' if k in own else '묶음')
per2 = RIx2.drop_duplicates(['company', 'title']).pivot_table(index='company', columns='from', values='title', aggfunc='count').fillna(0).astype(int)
CT2 = pd.read_csv(SEARCH / 'company_types.csv', dtype=str).set_index('name')
per2 = per2.join(CT2[['base_type', 'weak']])
print('■ 종목별 기사 수(3일, 제목 중복 제거)')
display(per2.sort_values('weak'))
print('■ 제목 표본(종류마다 검색 문자열 3개, 상위 3건)')
rng = random.Random(7)
for kind, g in uq[uq['n'] > 0].groupby('kind'):
    for q in rng.sample(sorted(set(g['query'])), min(3, g['query'].nunique())):
        print(f"[{kind}] {q} ({int(RQ2.set_index('query').at[q, 'n'])}건)")
        for it in RI2[RI2['query'] == q].head(3).itertuples():
            print(f'    - {it.title[:110]}')

■ 종류·모양별(기간 안, 본문 확보 매체)


검색  기사수_중앙값  영건_비율
kind   shape                      
구체 제품  주제만      41      1.0   0.49
       회사 + 주제  42      0.0   0.83
넓은 제품  주제만      14     25.5   0.14
       회사 + 주제  16      0.0   0.69
브랜드    이름만      17      0.0   0.65
       회사 + 주제  17      0.0   0.76
업종 기본  주제만      24      0.0   0.67
원자재·지표 주제만       7     32.0   0.00
       회사 + 주제   8      0.0   0.88
자기 회사  이름만      20      5.5   0.15
정책     주제만       2      0.0   1.00
       회사 + 주제   2      0.0   1.00
테마 기본  주제만      40      0.0   0.62
테마 대표  주제만      20     12.0   0.10
       회사 + 주제  28      0.0   0.93
해외 상대  상대 + 제품   7      0.0   0.57
       이름만      20     64.5   0.05

■ 단어 수별(주제만)


,검색,기사수_중앙값,영건_비율
단어,,,
1,12,65.0,0.08
2,52,11.5,0.23
3,60,0.0,0.63
4,5,0.0,0.60
5,5,0.0,1.00


■ 종목별 기사 수(3일, 제목 중복 제거)


,묶음,회사 고유,base_type,weak
company,,,,
셀트리온,105,242,의약·바이오,False
펩트론,122,175,의약·바이오,False
크래프톤,16,1100,브랜드·소비자,False
우진,83,75,수주,False
한미반도체,34,228,제품·공급망,False
대한항공,234,1942,원자재·가격,False
한화에어로스페이스,502,97,수주,False
SK하이닉스,6,1572,제품·공급망,False
POSCO홀딩스,37,992,원자재·가격,False


■ 제목 표본(종류마다 검색 문자열 3개, 상위 3건)
[구체 제품] "animation" (65건)
    - Disney Embraces AI It Once Fought, From Animation to Finance and Film - Yahoo Finance
    - Will Social Security go broke in 2032? What retirees need to know. - Yahoo Finance
    - Clarion Partners’ loan on Netflix-occupied Hollywood campus hits market - Yahoo Finance
[구체 제품] "NONGSHIM" "juice" (1건)
    - Maeil Dairies Co., Ltd. (267980.KQ) Stock Price, News, Quote & History - Yahoo Finance
[구체 제품] "calibration services" (10건)
    - Boyd Group Services Inc. Reports Third Quarter 2026 Results - Yahoo! Finance Canada
    - Boyd Group Services Inc. (BGSI) Q3 FY2026 earnings call transcript - Yahoo Finance
    - What Boyd Group Services (BYD) Said on Its Q3 Earnings Call - Yahoo Finance
[넓은 제품] "HD KOREA SHIPBUILDING & OFFSHORE ENGINEERING" "research and development services" (1건)
    - HJ Shipbuilding & Construction Co., Ltd. (097230.KS) Stock Price, News, Quote & History - Yahoo Finance
[넓은 제품] "IT outsourcing" (11건)
    - In

## 8. 기사가 아닌 페이지 거르기

구글 뉴스 RSS 결과에는 원문 주소가 없고(구글 링크만 있음) **제목과 매체만** 있다. 그래서 거르기를 세 단계로 나눈다. 이 절은 1단계만 적용하고, 2·3단계 규칙은 본문 받기를 만들 때 쓰도록 옮겨 둔다.

| 단계 | 볼 수 있는 것 | 규칙 출처 |
|---|---|---|
| 1. 검색 결과(이 절) | 제목, 매체 주소(하위 도메인 포함) | `graph_db/news_filter.ipynb`의 제목 규칙(팟캐스트·사진, 선물 추천, 행사 홍보, 인사 동정) + 2차 시험에서 본 꼴(시세·차트 페이지, 실적 발표 대본, 빈 제목) + 언론사 판정의 제외 구역 중 호스트 규칙(USA Today 팬 사이트·여행 목록, Business Insider 보도자료, CNN 방송 대본) |
| 2. 원문 주소 복원 뒤 | 원문 주소 | `news_source_tests/1_google_rss_source_test.ipynb`의 영상·라이브 경로와 매체별 제외 구역(`SECTION_EXCLUDE_RULES`), `news_filter`의 경로 규칙, 야후 시세 페이지(`/quote/`) |
| 3. 본문을 받은 뒤 | 페이지 HTML | `1_google_rss_source_test.ipynb`의 `classify_page`: JSON-LD 기사 유형·작성 시각이 없으면 기사 아님, 보도자료, 유료벽, 본문 짧음 |

제목 규칙은 제목 끝의 " - 매체 이름"을 뗀 뒤 대소문자를 가리지 않고 찾는다.

In [11]:
# ── 1단계: 제목 규칙(사유, 정규식). 2·3단계 규칙은 본문 받기를 만들 때 쓴다 ──
import re

TITLE_RULES = [
    # 2차 시험에서 본 꼴: 시세·차트·자료 페이지
    ('시세·자료 페이지', r'stock price, news, quote|quote & history|price history|seasonal returns chart|interactive stock chart|stock forum'
                       r'|analyst insights|price targets & recommendations|historical prices|options chain|key statistics|stock news today|\bstock performance$'),
    ('실적 발표 대본', r'earnings call transcript|call transcript$|prepared remarks'),
    ('빈 제목', r'(?-i:^[A-Z0-9]+(?:_[A-Z0-9]+)+$)'),                 # META_TITLE_QUOTE 같은 틀 이름(대소문자를 가린다)
    # graph_db/news_filter.ipynb의 제목 규칙
    ('영상·팟캐스트·사진', r'^podcast:|^in pictures:| in photos$|^in the know \d+'),
    ('생활·쇼핑', r'best .*gifts?\b|gift sets'),
    ('매체 홍보·행사 안내', r'^register:|final call for entries|webinar|free ebook|^get the .* newsletter'),
    ('인사 동정 모음', r'^people moves|^people on the move|^jobs roundup'),
]
TITLE_RX = [(reason, re.compile(rx, re.I)) for reason, rx in TITLE_RULES]
# 2단계(원문 주소) 규칙: 본문 받기에서 쓴다
URL_RULES = {
    '영상·라이브': r'/(video|videos|watch|clip|clips|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)|/(live|live-news|live-updates|liveblog)(/|$)',
    '시세·자료 페이지': r'finance\.yahoo\.com/quote/|/price-of-|/industry-statistics/|/document/',
    '외부 기고·홍보': r'/opinion/|/reader-forum/|^/newsletters/|supplier-spotlight',
}
SECTION_EXCLUDE_RULES = {   # 1_google_rss_source_test.ipynb와 같다(매체별 제외 구역, "호스트/경로"에서 찾음)
    'businessinsider.com': [r'^markets\.', r'^africa\.'], 'usatoday.com': [r'^ftw\.', r'^patriotswire\.', r'/story/shopping/', r'/press-release/'],
    'cnbc.com': [r'/investingclub/'], 'cnn.com': [r'^transcripts\.', r'/cnn-underscored/', r'^cnnicevents\.'], 'nypost.com': [r'/betting/'],
}
# 1단계에도 쓸 수 있는 호스트 규칙: 검색 결과의 매체 주소(하위 도메인 포함)로 거른다(SECTION_EXCLUDE_RULES 중 호스트 규칙 + 2차 시험에서 본 것)
HOST_RULES = [
    ('스포츠 팬 사이트', r'^ftw\.usatoday\.com$|wire\.usatoday\.com$'),
    ('여행·생활 목록', r'^10best\.usatoday\.com$'),
    ('보도자료', r'^markets\.businessinsider\.com$'),
    ('방송 대본·행사', r'^transcripts\.cnn\.com$|^cnnicevents\.'),
    ('지역판 복사본', r'^africa\.businessinsider\.com$'),
]
HOST_RX = [(reason, re.compile(rx, re.I)) for reason, rx in HOST_RULES]
strip_source = lambda t: re.sub(r'\s+-\s+[^-]+$', '', t or '').strip()


def title_reason(title, domain=''):
    t = strip_source(title)
    return (next((reason for reason, rx in HOST_RX if rx.search(domain or '')), '')
            or next((reason for reason, rx in TITLE_RX if rx.search(t)), ''))


RI2['drop'] = [title_reason(t, d) for t, d in zip(RI2['title'], RI2['domain'])]
u = RI2.drop_duplicates('title')
print(f"2차 시험 기사 {len(u):,}건(제목 중복 제거) 중 걸러짐 {(u['drop'] != '').sum():,}건({(u['drop'] != '').mean():.0%})", u['drop'].value_counts().drop('', errors='ignore').to_dict())
dom = u['domain'].str.replace(r'^(uk|ca|au|sg|nz|in)\.', '', regex=True)
print('매체별 걸러진 비율', u.assign(d=dom).groupby('d')['drop'].agg(lambda s: round((s != '').mean(), 2)).sort_values(ascending=False).head(8).to_dict())
print('■ 걸러진 제목 예')
for reason, g in u[u['drop'] != ''].groupby('drop'):
    print(f'  [{reason}]', [strip_source(t)[:70] for t in g['title'].head(3)])
print('■ 남은 야후 제목 예', [strip_source(t)[:70] for t in u[(u['drop'] == '') & u['domain'].str.contains('yahoo')]['title'].sample(6, random_state=3)])

2차 시험 기사 6,833건(제목 중복 제거) 중 걸러짐 1,569건(23%) {'시세·자료 페이지': 906, '실적 발표 대본': 335, '보도자료': 133, '스포츠 팬 사이트': 103, '지역판 복사본': 75, '생활·쇼핑': 5, '빈 제목': 3, '방송 대본·행사': 3, '여행·생활 목록': 3, '매체 홍보·행사 안내': 3}
매체별 걸러진 비율 {'10best.usatoday.com': 1.0, 'ftw.usatoday.com': 1.0, 'giantswire.usatoday.com': 1.0, 'africa.businessinsider.com': 1.0, 'jaguarswire.usatoday.com': 1.0, 'jetswire.usatoday.com': 1.0, 'markets.businessinsider.com': 1.0, 'ninerswire.usatoday.com': 1.0}
■ 걸러진 제목 예
  [매체 홍보·행사 안내] ['From Data to Discovery: Find the Right Patients Across the Therapy Lif', "Join FibroBiologics' Live Investor Webinar: Advancing a Fibroblast-Bas", "Join FibroBiologics' Live Investor Webinar: Advancing a Fibroblast-Bas"]
  [방송 대본·행사] ['CNN.com - Transcripts', 'CNN Transcripts for November 13, 2026', 'CNN News Central']
  [보도자료] ['Cloudforce Named 2026 Microsoft Education Partner of the Year, the Fir', 'ALSO and Intermedia Form Joint Venture to Build a Next-Generation U.S.', 'SHI Recognized as the Winner of

In [12]:
# ── 거른 뒤 다시 보기: 종류·모양별, 단어 수별, 종목별 ──
kept = RI2[RI2['drop'] == '']
n_kept = kept.groupby('query')['title'].nunique()
R3 = TESTQ.drop_duplicates(['kind', 'shape', 'query']).assign(n=lambda d: d['query'].map(n_kept).fillna(0).astype(int))
print('■ 종류·모양별(거른 뒤)')
display(R3.groupby(['kind', 'shape']).agg(검색=('query', 'size'), 기사수_중앙값=('n', 'median'), 영건_비율=('n', lambda s: round((s == 0).mean(), 2))))
t3 = R3[R3['shape'] == '주제만'].drop_duplicates('query').copy()
t3['단어'] = t3['term'].str.split().str.len().clip(upper=5)
print('■ 단어 수별(주제만, 거른 뒤)')
display(t3.groupby('단어').agg(검색=('query', 'size'), 기사수_중앙값=('n', 'median'), 영건_비율=('n', lambda s: round((s == 0).mean(), 2))))
Kx = kept.merge(TESTQ[['query', 'company', 'kind']], on='query')
Kx['from'] = Kx['kind'].map(lambda k: '회사 고유' if k in own else '묶음')
per3 = Kx.drop_duplicates(['company', 'title']).pivot_table(index='company', columns='from', values='title', aggfunc='count').fillna(0).astype(int)
print('■ 종목별 기사 수(거른 뒤, 3일)')
display(per3.join(CT2[['base_type', 'weak']]).sort_values('weak'))
print('kept_items.csv', save_csv(kept, OUT / SECOND / 'kept_items.csv'))

■ 종류·모양별(거른 뒤)


검색  기사수_중앙값  영건_비율
kind   shape                      
구체 제품  주제만      41      0.0   0.51
       회사 + 주제  42      0.0   0.88
넓은 제품  주제만      14      6.0   0.21
       회사 + 주제  16      0.0   0.88
브랜드    이름만      17      0.0   0.71
       회사 + 주제  17      0.0   0.76
업종 기본  주제만      24      0.0   0.71
원자재·지표 주제만       7     17.0   0.00
       회사 + 주제   8      0.0   0.88
자기 회사  이름만      20      1.0   0.35
정책     주제만       2      0.0   1.00
       회사 + 주제   2      0.0   1.00
테마 기본  주제만      40      0.0   0.62
테마 대표  주제만      20      6.0   0.25
       회사 + 주제  28      0.0   0.96
해외 상대  상대 + 제품   7      0.0   0.57
       이름만      20     45.0   0.05

■ 단어 수별(주제만, 거른 뒤)


,검색,기사수_중앙값,영건_비율
단어,,,
1,12,41.0,0.08
2,52,5.0,0.25
3,60,0.0,0.70
4,5,0.0,0.80
5,5,0.0,1.00


■ 종목별 기사 수(거른 뒤, 3일)


,묶음,회사 고유,base_type,weak
company,,,,
셀트리온,75,178,의약·바이오,False
펩트론,94,118,의약·바이오,False
크래프톤,4,964,브랜드·소비자,False
우진,54,53,수주,False
한미반도체,29,168,제품·공급망,False
대한항공,182,1611,원자재·가격,False
한화에어로스페이스,379,52,수주,False
SK하이닉스,6,1464,제품·공급망,False
POSCO홀딩스,22,580,원자재·가격,False


kept_items.csv 그대로


## 9. 3차 시험: 2단어 중심 검색어

2차 시험에서 검색어가 길수록(3단어 이상 대부분 0건), "회사 + 주제" 모양(88% 0건)일수록 기사가 안 잡히고, 일반 영어 한 단어(juice, ACROSS)는 상관없는 기사가 쏟아졌다. 그래서 `graph_db/search_terms`에서 규칙을 바꿔 시험 목록을 다시 만들었다(`graph_db/graphDB.md` 6.5).

- 업종·테마·회사 기본 검색어: 지시문 `search_base_v4`·`search_company_v2`(2단어 기본, 3단어까지)
- 후보: 일반 영어 한 단어 제외(원자재·약칭·회사 이름은 둠), 3자 이하 회사 이름은 긴 이름(LX → LX Holdings, GKL → Grand Korea Leisure)
- 모양: "회사 + 주제" 뺌, 3단어 이상 주제는 따옴표 없는 모양도

같은 매체 11곳·같은 기간(11-12~14)·같은 방식(야후 1건 + 나머지 OR 1건, 한도면 나눔). **2차에 있던 검색 문자열은 2차 응답을 그대로 쓰고**, 새 것만 요청한다. 결과는 8절의 1단계 거르기를 적용해 센다.

In [13]:
# ── 3차 시험: 새 검색 문자열만 요청(2차에 있던 것은 2차 응답을 쓴다) ──
THIRD = 'rss_test_short'
TESTQ3 = frozen_testq(THIRD)
terms3 = list(dict.fromkeys(TESTQ3['query']))
src_of = lambda t: SECOND if raw_path(SECOND, y_q(t)).exists() and raw_path(SECOND, r_q(t)).exists() else THIRD
new3 = [t for t in terms3 if src_of(t) == THIRD]
print(f'3차 시험 검색 문자열 {len(terms3)}개 / 2차 응답을 쓰는 것 {len(terms3) - len(new3)} / 새로 받을 것 {len(new3)}(요청 {2 * len(new3)}건)')
fetch_all(THIRD, [q for t in new3 for q in (y_q(t), r_q(t))])
split3 = [q for t in new3 for q in split_queries(THIRD, t)]
print(f'한도에 걸려 나눌 요청 {len(split3)}건')
fetch_all(THIRD, split3)

3차 시험 검색 문자열 213개 / 2차 응답을 쓰는 것 134 / 새로 받을 것 79(요청 158건)
rss_test_short: 검색 158개 / 받은 것 158 / 남은 것 0 (예상 0분)


한도에 걸려 나눌 요청 100건
rss_test_short: 검색 100개 / 받은 것 100 / 남은 것 0 (예상 0분)


In [14]:
# ── 3차 시험 결과 모으기 + 1단계 거르기 ──
rows3, items3 = [], []
for t in terms3:
    g = gather(src_of(t), t)
    if g is None:
        continue
    seen, capped = g
    items3 += [dict(it, query=t) for it in seen.values()]
    rows3.append(dict(query=t, n=len(seen), capped=capped, yahoo=sum('yahoo' in it['domain'] for it in seen.values()), from_second=src_of(t) == SECOND))
RQ3, RI3 = pd.DataFrame(rows3), pd.DataFrame(items3)
if len(RQ3) == len(terms3):
    RI3['drop'] = [title_reason(t, d) for t, d in zip(RI3['title'], RI3['domain'])]
    kept3 = RI3[RI3['drop'] == '']
    RQ3['kept'] = RQ3['query'].map(kept3.groupby('query')['title'].nunique()).fillna(0).astype(int)
    print(f"검색어 {len(RQ3)} / 기사 {RI3['title'].nunique():,}건 → 거른 뒤 {kept3['title'].nunique():,}건 / 0건(거른 뒤) {(RQ3['kept'] == 0).sum()}개 / 한도에 걸려 나눈 것 {RQ3['capped'].sum()}개")
    print('queries.csv', save_csv(RQ3, OUT / THIRD / 'queries.csv'), '/ items.csv', save_csv(RI3, OUT / THIRD / 'items.csv'),
          '/ kept_items.csv', save_csv(kept3, OUT / THIRD / 'kept_items.csv'))
else:
    print(f'아직 다 받지 못했다({len(RQ3)}/{len(terms3)}). RUN = True로 위 칸을 실행한다.')

검색어 213 / 기사 8,593건 → 거른 뒤 6,447건 / 0건(거른 뒤) 66개 / 한도에 걸려 나눈 것 20개
queries.csv 그대로 / items.csv 그대로 / kept_items.csv 그대로


In [15]:
# ── 3차 시험 결과: 2차와 비교(종류·모양별, 단어 수별, 따옴표 있음·없음, 기본 검색어 v3·v4, 종목별) ──
stat = lambda d: d.agg(검색=('query', 'size'), 기사수_중앙값=('n', 'median'), 영건_비율=('n', lambda s: round((s == 0).mean(), 2)), 기사수_합=('n', 'sum'))
if len(RQ3) == len(terms3):
    R4 = TESTQ3.drop_duplicates(['kind', 'shape', 'query']).assign(n=lambda d: d['query'].map(RQ3.set_index('query')['kept']).fillna(0).astype(int))
    print('■ 종류·모양별(거른 뒤): 3차')
    display(stat(R4.groupby(['kind', 'shape'])))
    print('■ 종류별(거른 뒤, 주제만·이름만): 2차 → 3차')
    one = lambda R: R[R['shape'].isin(['주제만', '이름만'])].groupby('kind')['n'].agg(['size', 'median'])
    display(one(R3).join(one(R4), lsuffix='_2차', rsuffix='_3차'))
    t4 = R4[R4['shape'] == '주제만'].drop_duplicates('query').copy()
    t4['단어'] = t4['term'].str.split().str.len().clip(upper=5)
    print('■ 단어 수별(주제만, 거른 뒤): 3차')
    display(stat(t4.groupby('단어')))
    # 따옴표 있음·없음: 3단어 이상 같은 주제
    qn = R4[R4['shape'].str.startswith('주제만')].pivot_table(index='term', columns='shape', values='n', aggfunc='max').dropna()
    if len(qn):
        print(f"■ 3단어 이상 주제 {len(qn)}개: 따옴표 있음 중앙값 {qn['주제만'].median():.0f}(0건 {(qn['주제만'] == 0).mean():.0%}) / 없음 {qn['주제만·따옴표 없음'].median():.0f}(0건 {(qn['주제만·따옴표 없음'] == 0).mean():.0%})")
        display(qn.sort_values('주제만·따옴표 없음', ascending=False))
        print('■ 따옴표 없는 검색의 제목 예(관련 있는지 보기)')
        for term in qn.sort_values('주제만·따옴표 없음', ascending=False).index[:6]:
            print(f'  {term}: 따옴표 없음 {int(qn.at[term, "주제만·따옴표 없음"])}건')
            for it in kept3[kept3['query'] == term].head(4).itertuples():
                print(f'    - {strip_source(it.title)[:110]}')
    # 기본 검색어: 2차(v3) → 3차(v4)
    base_k = ['업종 기본', '테마 기본', '회사 기본']
    b2 = R3[R3['kind'].isin(base_k) & (R3['shape'] == '주제만')].assign(단어=lambda d: d['term'].str.split().str.len())
    b3 = R4[R4['kind'].isin(base_k) & (R4['shape'] == '주제만')].assign(단어=lambda d: d['term'].str.split().str.len())
    print('■ 기본 검색어(주제만, 거른 뒤): 2차(v3·회사 v1) → 3차(v4·회사 v2)')
    display(pd.DataFrame({'2차': [len(b2), b2['단어'].mean().round(2), b2['n'].median(), round((b2['n'] == 0).mean(), 2)],
                          '3차': [len(b3), b3['단어'].mean().round(2), b3['n'].median(), round((b3['n'] == 0).mean(), 2)]},
                         index=['검색', '평균 단어 수', '기사수 중앙값', '0건 비율']))
    print('  3차 기본 검색어:', ', '.join(f"{r.term}({r.n})" for r in b3.drop_duplicates('term').itertuples()))
    # 종목별: 2차 → 3차(거른 뒤, 제목 중복 제거)
    K4 = kept3.merge(TESTQ3[['query', 'company', 'kind']], on='query')
    K4['from'] = K4['kind'].map(lambda k: '회사 고유' if k in own else '묶음')
    per4 = K4.drop_duplicates(['company', 'title']).pivot_table(index='company', columns='from', values='title', aggfunc='count').fillna(0).astype(int)
    print('■ 종목별 기사 수(거른 뒤, 3일): 2차 → 3차')
    display(per3.join(per4, lsuffix='_2차', rsuffix='_3차').fillna(0).astype(int).join(CT2[['base_type', 'weak']]).sort_values('weak'))

■ 종류·모양별(거른 뒤): 3차


검색  기사수_중앙값  영건_비율  기사수_합
kind   shape                                
구체 제품  주제만         40      0.0   0.60    322
       주제만·따옴표 없음  18     18.0   0.11    928
넓은 제품  주제만         14      6.0   0.21    546
       주제만·따옴표 없음   4    331.5   0.00   1382
브랜드    이름만         17      0.0   0.76      4
업종 기본  주제만         25      7.0   0.12    515
원자재·지표 주제만          7     17.0   0.00    181
       주제만·따옴표 없음   1    123.0   0.00    123
자기 회사  이름만         19      1.0   0.37    179
정책     주제만          2      0.0   1.00      0
       주제만·따옴표 없음   2    247.0   0.00    494
테마 기본  주제만         38      6.0   0.18    599
       주제만·따옴표 없음   2    247.0   0.00    494
테마 대표  주제만         13      3.0   0.08    148
       주제만·따옴표 없음   4    129.5   0.00    547
해외 상대  상대 + 제품      7      0.0   0.57     81
       이름만         20     45.0   0.05   3067
회사 기본  주제만          2     23.5   0.50     47

■ 종류별(거른 뒤, 주제만·이름만): 2차 → 3차


,size_2차,median_2차,size_3차,median_3차
kind,,,,
구체 제품,41,0.0,40,0.0
넓은 제품,14,6.0,14,6.0
브랜드,17,0.0,17,0.0
업종 기본,24,0.0,25,7.0
원자재·지표,7,17.0,7,17.0
자기 회사,20,1.0,19,1.0
정책,2,0.0,2,0.0
테마 기본,40,0.0,38,6.0
테마 대표,20,6.0,13,3.0


■ 단어 수별(주제만, 거른 뒤): 3차


,검색,기사수_중앙값,영건_비율,기사수_합
단어,,,,
1,5,16.0,0.20,87
2,86,5.0,0.22,1713
3,22,0.0,0.55,109
4,4,0.0,0.75,5
5,4,0.0,1.00,0


... (2 outputs left out)

■ 따옴표 없는 검색의 제목 예(관련 있는지 보기)
  research and development services: 따옴표 없음 565건
    - Prediction: This Quantum Computing Stock Will Be Worth More Than IonQ 5 Years From Now
    - Applied Materials Announces Fourth Quarter and Fiscal Year 2026 Results
    - Could a Former Nvidia Executive Replace Jensen Huang? Arm’s Rene Haas Emerges as Candidate
    - Bitcoin to $1 Million by 2028? Blockstream CEO Adam Back Says Yes
  Approved List of Models and Manufacturers: 따옴표 없음 473건
    - 5 Multibagger Stocks to Buy According to Wall Street
    - Why Management Is Cautious About One Nebius (NBIS) Number
    - 5 Best Renewable & Clean Energy Stocks To Buy According to Analysts
    - Forget SpaceX: This Potential Blockbuster IPO Has Attracted $26 Billion From AI Giants
  app store fees: 따옴표 없음 458건
    - Apple users are about to lose something they've had for 28 years
    - Jeff Bezos predicts AI could let families live on one paycheck. Amodei and Gates warn of a rougher road
    - Apple (AAPL) Puts 

,2차,3차
검색,64.00,65.00
평균 단어 수,2.73,1.97
기사수 중앙값,0.00,6.00
0건 비율,0.66,0.17


  3차 기본 검색어: DRAM prices(3), HBM demand(4), memory prices(52), drug pricing(13), FDA approvals(8), biosimilar competition(6), biologics manufacturing(6), drug approvals(9), drug licensing(0), Korean food(5), grain prices(2), K-beauty(16), apparel sales(7), cosmetics tariffs(0), cleaning products(5), palm oil(9), mobile games(13), digital advertising(67), app store fees(2), K-pop(31), concert tours(0), Korean content(1), streaming rights(1), Chinese tourists(1), outbound tourism(1), Macau casinos(0), LNG carriers(3), shipbuilding orders(0), Saudi megaprojects(0), LNG projects(3), defense spending(43), artillery shells(0), aircraft production(8), satellite launches(13), power transformers(5), grid investment(17), nuclear reactors(27), small modular reactors(19), steel tariffs(2), steel prices(19), LNG prices(8), thermal coal(7), freight rates(47), air cargo(27), international travel(22), jet fuel(51), Treasury yields(195), Fed policy(29), credit spreads(16), duty-free sales(0), display c

,묶음_2차,회사 고유_2차,묶음_3차,회사 고유_3차,base_type,weak
company,,,,,,
셀트리온,75,178,109,178,의약·바이오,False
펩트론,94,118,97,683,의약·바이오,False
크래프톤,4,964,470,923,브랜드·소비자,False
우진,54,53,78,410,수주,False
한미반도체,29,168,319,531,제품·공급망,False
대한항공,182,1611,114,307,원자재·가격,False
한화에어로스페이스,379,52,267,158,수주,False
SK하이닉스,6,1464,204,1462,제품·공급망,False
POSCO홀딩스,22,580,40,504,원자재·가격,False


### 3차 시험 결론 (2026-11-15)

새 검색 문자열 79개(요청 158건 + 한도로 나눈 100건, 6초 간격 약 27분, 막힘 없음). 거른 뒤 기사 6,447건.

- **2단어 기본 검색어가 효과가 컸다**: 업종·테마·회사 기본 검색어 중앙값 0 → 6건, 0건 비율 66% → 17%(평균 2.73 → 1.97단어). 종목별 묶음 검색어 기사: SK하이닉스 6 → 35, 크래프톤 4 → 61, 티에프이 34 → 76, LX홀딩스 0 → 46. 제목도 맞는다(memory prices → 마이크론·샌디스크 메모리 부족, K-beauty → K-뷰티 투자 붐). 다만 credit spreads처럼 거시 일반어는 상관없는 시장 기사가 섞인다.
- **단어 수**(주제만): 2단어 중앙값 5건(0건 22%), 3단어 0건(55%), 4단어 이상 거의 0건.
- **따옴표 없는 모양은 쓰지 않는다**: 3단어 이상 30개의 중앙값이 0 → 42건으로 늘었지만 대부분 상관없는 기사였다(research and development services 565건, Approved List of Models and Manufacturers 473건: 종목 추천·비트코인 기사). 긴 금융 기사에는 흔한 단어가 다 들어 있다.
- **일반 영어 한 단어를 빼자** 잡음이 사라졌다: 대한항공 회사 고유 1,611 → 287건(ACROSS), 농심 147 → 8(juice), 하이브 163 → 4(documentaries). 한화에어로스페이스는 테마 대표 drones·satellites가 빠져 379 → 65(방산과 관련 있는 말이라 기본 검색어에서 좁혀 쓰는 것이 낫다: 예 military drones).
- **짧은 회사 이름을 긴 이름으로**: Grand Korea Leisure, LX Holdings는 0건(LX 12건은 대부분 Lexus LX였다). 중소형 한국 회사 이름은 미국 매체에 거의 안 나온다. General Motors는 56건.
- **여전히 안 잡히는 것**: 구체 제품(중앙값 0, 60% 0건)과 브랜드(76% 0건). 제품 사전 이름이 길고 기술적이다(board-on-chip and chip-on-board bonders, handler change kits). → 남은 과제: 제품 노드에 짧은 검색용 이름을 붙이거나, 구체 제품은 검색어에서 빼고 업종·테마 기본 검색어에 맡긴다(`graph_db/graphDB.md` 결정할 것 #22).